In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
from pathlib import Path
import openslide
import PIL.Image
import tifffile
import re

import sys
sys.path.append("/workspaces/TRIDENT")

from trident.wsi_objects.utils import extract_tile_size

In [ ]:

def extract_magnification(tif_path: Path) -> float:
    with tifffile.TiffFile(tif_path) as tif:
        if "ImageDescription" not in tif.pages[0].tags:
            raise ValueError("ImageDescription tag not found")
        image_description = tif.pages[0].tags["ImageDescription"].value
        # find "|AppMag = *|" in the image description
        m = re.search(r"\|AppMag = ([0-9\.]+)\|", image_description)
        if m is None:
            print("ImageDescription:", image_description)
            raise ValueError("AppMag not found in ImageDescription")
        app_mag = float(m.group(1))
        return app_mag


def extract_mpp(tif_path: Path) -> float:
    with tifffile.TiffFile(tif_path) as tif:
        if "ImageDescription" not in tif.pages[0].tags:
            raise ValueError("ImageDescription tag not found")
        image_description = tif.pages[0].tags["ImageDescription"].value
        # find "|MPP = *|" in the image description
        m = re.search(r"\|MPP = ([0-9\.]+)\|", image_description)
        if m is None:
            print("ImageDescription:", image_description)
            raise ValueError("MPP not found in ImageDescription")
        mpp = round(float(m.group(1)), 2)
        return mpp


In [ ]:
slides_dir = Path("/mnt/oskar/slides")
slide_paths = list(slides_dir.glob("*.svs"))

tilesize_groups = dict()
for slide_path in slide_paths:
    native_tile_size = extract_tile_size(slide_path)
    mag = extract_magnification(slide_path)
    mpp = extract_mpp(slide_path)
    key = f"{native_tile_size}_{mag}_{mpp}"
    tilesize_groups[key] = tilesize_groups.get(key, 0) + 1

for key, count in tilesize_groups.items():
    tilesize, mag, mpp = key.split("_")
    print(f"Tile size: {tilesize}, Magnification: {mag}, MPP: {mpp}, Count: {count}")


In [ ]:
import h5py

def extract_total_selected_tile_bytecount(h5_path: Path) -> int:
    with h5py.File(h5_path, "r") as f:
        bytecounts = f["bytecounts"][:]
    
    # if bytecounts.shape[1] == 1:
    #     return 0, 0
    # add up along second axis
    bytecounts = np.sum(bytecounts, axis=1)
    # select 50% largest elements
    num_elements = bytecounts.shape[0]
    num_elements_to_select = int(num_elements * 1.0)
    indices = np.argpartition(bytecounts, -num_elements_to_select)[-num_elements_to_select:]
    # select the largest elements
    bytecounts = bytecounts[indices]
    # sum
    total_bytecount = np.sum(bytecounts)
    # convert to MB
    return total_bytecount, num_elements_to_select

patches_files = Path("/mnt/oskar/40x_512px_0px_overlap/patches").glob("*.h5")
total_bytecount = 0
total_tiles = 0
for h5_path in patches_files:
    bytecount, num_tiles = extract_total_selected_tile_bytecount(h5_path)
    total_bytecount += bytecount
    total_tiles += num_tiles

# convert to GB
total_bytecount_gb = total_bytecount / (1024**3)
print(f"Total bytecount: {total_bytecount_gb} GiB")
print(f"Total tiles: {total_tiles}")

# native tile size: 512
# Total bytecount: 265.91337614133954 GiB
# Total tiles: 4536689

# native tile size: 256
# Total bytecount: 375.42709517572075 GiB
# Total tiles: 6142864

In [ ]:
segment_again = ["f2024001766t1-a-6_130955",
"h2019018560t5-a-2_161036",
"h2022003279t1-a-3_032528",
"h2022005119t1-a-4_093353",
"h2022007825t1-a-8_233639",
"h2023010049t1-a-5_111127",
"h2024002581t11-a-3_121401",
"h2024007715t2-a-2_102433",
"h2024012704t1-a-2_110442",
"h2024017094t2-a-3_132305",
"n2020001261t1-a-3_190650",
"n2022001164t1-a-3_124229",
"n2023001964t3-a-4_113847"]
missing = ["e2024000032t1-a-4_143250"]
#? empty? h2021014634t2-a-5_023026

contours_dir = Path("/mnt/oskar/contours")
geo_contours_dir = Path("/mnt/oskar/contours_geojson")
thumbnails_dir = Path("/mnt/oskar/thumbnails")
for slide_name in segment_again:
    # remove the three files in the directories having this slide name
    contours_path = contours_dir / f"{slide_name}.jpg"
    geo_contours_path = geo_contours_dir / f"{slide_name}.geojson"
    thumbnail_path = thumbnails_dir / f"{slide_name}.jpg"
    # if contours_path.exists():
    #     contours_path.unlink()
    # if geo_contours_path.exists():
    #     geo_contours_path.unlink()
    # if thumbnail_path.exists():
    #     thumbnail_path.unlink()

In [ ]:
def show_slide_infos(tif_path: Path, level: int):
    # only count pages that are tiled
    page_counter = 0
    with tifffile.TiffFile(tif_path) as tif:
        for page in tif.pages:
            if "TileWidth" not in page.tags:
                continue
            if page_counter == level:
                print(f"Page {page_counter} shape: {page.shape}")
                print(page.tags)
                num_tiles = len(page.tags["TileOffsets"].value)
                print(f"Page {page_counter} Total Tiles: {num_tiles}")
                return
            elif page_counter > level:
                break
            else:
                page_counter += 1
    raise ValueError(f"Page {level} not found")

In [ ]:
empty_slide = "/mnt/oskar/slides/h2025001976t1-a-4_102918.svs"
show_slide_infos(empty_slide, 1)
slide = openslide.OpenSlide(empty_slide)

thumbnail = slide.get_thumbnail((2048, 2048))
thumbnail = thumbnail.convert("RGB")

fig, ax = plt.subplots(figsize=(20, 10))
ax.imshow(thumbnail)
ax.axis("off")
plt.show()

